# T-seeded DD Startup Verification

This notebook demonstrates the **T-seeded DD startup** model using a **step-by-step approach** with functions imported from the repository.

The T-seeded model tracks tritium inventory evolution through four state variables:
- **N_ofc**: outer-fuel cycle tritium inventory (atoms)
- **N_ifc**: Inner-fuel cycle tritium inventory (atoms)
- **N_stor**: Storage tritium inventory (atoms)
- **n_T**: Tritium density in plasma (m⁻³)

The model integrates coupled ODEs until D-T equilibrium is reached (n_T = 0.5*n_tot).

## 1. Import Functions from Repository

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import solver from repository
from src.physics.Tseeded_functions import solve_ode_system

# Import reactivity functions
from src.physics.reactivity_functions import (
    sigmav_DD_p_BoschHale,
    sigmav_DD_n_BoschHale,
    sigmav_DT_BoschHale
)

# Import registry constants
from src.registry.parameter_registry import lambda_T, tritium_mass, REACTION_ENERGY_BY_CHANNEL

## 2. Define Reactor Parameters

In [ ]:
# Reactor geometry and plasma parameters
V_plasma = 1500.0  # Plasma volume (m³)
n_tot = 1.5e20     # Total particle density (m⁻³)
T = 15.0           # Plasma temperature (keV)

# Confinement and breeding parameters
tau_p_T = 5.0      # Tritium particle confinement time (s)
TBR_DT = 1.05      # D-T tritium breeding ratio
TBR_DDn = 0.9      # DD neutron tritium breeding ratio

# Fuel cycle processing times
tau_ifc = 100.0    # Inner-fuel cycle processing time (s)
tau_ofc = 24*3600  # outer-fuel cycle processing time (s, ~1 day)

# Injection parameters
injection_rate_max = 1e22  # Maximum tritium injection rate (atoms/s)
N_stor_min = 0.001 / tritium_mass  # Minimum stored tritium (atoms, default: 1g)

# Simulation time limit
max_simulation_time = 10 * 365 * 24 * 3600  # 10 years

print(f"Reactor Parameters:")
print(f"  V_plasma = {V_plasma} m³")
print(f"  n_tot = {n_tot:.2e} m⁻³")
print(f"  T = {T} keV")
print(f"  tau_p_T = {tau_p_T} s")
print(f"  TBR_DT = {TBR_DT}")
print(f"  TBR_DDn = {TBR_DDn}")
print(f"  tau_ifc = {tau_ifc} s")
print(f"  tau_ofc = {tau_ofc/(24*3600):.1f} days")
print(f"  injection_rate_max = {injection_rate_max:.2e} atoms/s")
print(f"  N_stor_min = {N_stor_min:.2e} atoms ({N_stor_min*tritium_mass*1000:.1f} g)")

## 3. Calculate Fusion Reactivities

Use Bosch-Hale parametrizations for DD and D-T reactions.

In [ ]:
# Calculate reactivities at plasma temperature
sigmav_DD_p = sigmav_DD_p_BoschHale(T)
sigmav_DD_n = sigmav_DD_n_BoschHale(T)
sigmav_DT = sigmav_DT_BoschHale(T)

print(f"Fusion Reactivities at T = {T} keV:")
print(f"  <σv>_DD_p = {sigmav_DD_p:.4e} m³/s")
print(f"  <σv>_DD_n = {sigmav_DD_n:.4e} m³/s")
print(f"  <σv>_DT   = {sigmav_DT:.4e} m³/s")
print(f"\nReactivity ratios:")
print(f"  <σv>_DT / <σv>_DD_p = {sigmav_DT/sigmav_DD_p:.1f}")
print(f"  <σv>_DT / <σv>_DD_n = {sigmav_DT/sigmav_DD_n:.1f}")

## 4. Run ODE Solver

In [ ]:
# Call the T-seeded ODE solver from repository
result = solve_ode_system(
    V_plasma=V_plasma,
    n_tot=n_tot,
    tau_p_T=tau_p_T,
    TBR_DT=TBR_DT,
    TBR_DDn=TBR_DDn,
    tau_ifc=tau_ifc,
    tau_ofc=tau_ofc,
    sigmav_DD_p=sigmav_DD_p,
    sigmav_DD_n=sigmav_DD_n,
    sigmav_DT=sigmav_DT,
    injection_rate_max=injection_rate_max,
    max_simulation_time=max_simulation_time,
    N_stor_min=N_stor_min
)

# Extract results
t = result['t']
N_ofc = result['N_ofc']
N_ifc = result['N_ifc']
N_stor = result['N_stor']
n_T = result['n_T']
t_startup = result['t_startup']
sol_success = result['sol_success']

# Print results
if sol_success:
    print(f"✓ Solver converged successfully")
    print(f"  Startup time: {t_startup:.2e} s ({t_startup/(365*24*3600):.3f} years)")
    print(f"  Number of time points: {len(t)}")
else:
    print(f"✗ Solver failed: {result.get('error', 'Unknown error')}")

## 5. Analyze Time Evolution

Calculate key metrics throughout the startup phase.

In [ ]:
# Convert time to days
t_days = t / (24 * 3600)

# Convert tritium inventories to kg
N_ofc_kg = N_ofc * tritium_mass
N_ifc_kg = N_ifc * tritium_mass
N_stor_kg = N_stor * tritium_mass
N_total_kg = N_ofc_kg + N_ifc_kg + N_stor_kg

# Calculate tritium fraction in plasma
f_T = n_T / n_tot

# Calculate injection rate
injection_rate = np.zeros_like(t)
for i in range(len(t)):
    if N_stor[i] > N_stor_min:
        inj_rate = N_ifc[i] / tau_ifc - lambda_T * N_stor[i]
        injection_rate[i] = max(0.0, min(inj_rate, injection_rate_max))

print(f"Final State (at {t_days[-1]:.2f} days):")
print(f"  Total tritium inventory: {N_total_kg[-1]:.3f} kg")
print(f"    - outer-fuel cycle: {N_ofc_kg[-1]:.3f} kg")
print(f"    - In-fuel cycle: {N_ifc_kg[-1]:.3f} kg")
print(f"    - Storage: {N_stor_kg[-1]:.3f} kg")
print(f"  Tritium fraction in plasma: {f_T[-1]:.3f}")
print(f"  Tritium density: {n_T[-1]:.2e} m⁻³")

## 6. Visualize Results

In [ ]:
# Create comprehensive visualization
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Tritium inventories evolution
ax = axs[0, 0]
ax.plot(t_days, N_ofc_kg, label='outer-fuel cycle', linewidth=2)
ax.plot(t_days, N_ifc_kg, label='In-fuel cycle', linewidth=2)
ax.plot(t_days, N_stor_kg, label='Storage', linewidth=2)
ax.plot(t_days, N_total_kg, label='Total', linewidth=2, linestyle='--', color='black')
ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Tritium Inventory (kg)', fontsize=12)
ax.set_title('Tritium Inventory Evolution', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Tritium fraction in plasma
ax = axs[0, 1]
ax.plot(t_days, f_T, linewidth=2, color='red')
ax.axhline(y=0.5, color='green', linestyle='--', linewidth=2, label='DT equilibrium')
ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Tritium Fraction f_T', fontsize=12)
ax.set_title('Tritium Fraction in Plasma', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 0.6])

# Plot 3: Injection rate
ax = axs[1, 0]
ax.plot(t_days, injection_rate / 1e20, linewidth=2, color='purple')
ax.axhline(y=injection_rate_max/1e20, color='orange', linestyle='--', linewidth=2, label='Maximum rate')
ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Injection Rate (×10²⁰ atoms/s)', fontsize=12)
ax.set_title('Tritium Injection Rate', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: Tritium density
ax = axs[1, 1]
ax.plot(t_days, n_T / 1e19, linewidth=2, color='blue')
ax.axhline(y=0.5*n_tot/1e19, color='green', linestyle='--', linewidth=2, label='DT equilibrium')
ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Tritium Density (×10¹⁹ m⁻³)', fontsize=12)
ax.set_title('Tritium Density Evolution', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary

This notebook demonstrated the **T-seeded DD startup** model using a **step-by-step approach**:

1. **Imported repository functions**: `solve_ode_system`, reactivity functions, registry constants
2. **Defined reactor parameters**: geometry, plasma conditions, fuel cycle parameters
3. **Calculated reactivities**: Used Bosch-Hale parametrizations for DD and D-T reactions
4. **Ran ODE solver**: Integrated coupled ODEs for tritium inventory evolution
5. **Analyzed time evolution**: Tracked inventories, tritium fraction, injection rate
6. **Visualized results**: Four-panel plot showing key metrics throughout startup

### Key Findings:

- **Startup time**: The time to reach D-T equilibrium (f_T = 0.5) depends on:
  - Tritium breeding ratios (TBR_DT, TBR_DDn)
  - Fuel cycle processing times (tau_ifc, tau_ofc)
  - Injection rate limits
  - Plasma confinement (tau_p_T)

- **Tritium inventory**: Total tritium accumulates in three subsystems:
  - outer-fuel cycle (breeding from DD and D-T reactions)
  - In-fuel cycle (recycling from plasma exhaust)
  - Storage (ready for injection)

- **Control strategy**: Injection rate is limited by:
  - Available storage inventory (N_stor > N_stor_min)
  - Maximum injection rate (injection_rate_max)
  - Balance between IFC processing and storage decay

### Next Steps:

- Compare with **lump model** (lumped tritium inventory)
- Explore **multispecies models** (tracking all isotopes)
- Perform **parametric studies** to optimize startup strategy
- Calculate **economic metrics** (startup cost, tritium requirements)